# Load and install required packages


We will be using the python package: choicellm for our model calls to remove a lot of code specific to initializing a model and working with a smaller model as we do here.

However, know that this can just as well be done without this package

In [1]:
%pip install -qU choicellm krippendorff

In [40]:
import pandas as pd
import json
import numpy as np
from sklearn.metrics import classification_report
import krippendorff
from sklearn.preprocessing import LabelEncoder

Clone the workshop repository to get the required files

It contains a baseline prompt, a validation .csv, and a .csv with some example transcripts you could use to improve your prompt

In [3]:
!git clone https://github.com/Kevinoost1/aps26_llm_ws.git

fatal: destination path 'aps26_llm_ws' already exists and is not an empty directory.


In [24]:
%cd aps26_llm_ws
!git pull

[Errno 2] No such file or directory: 'aps26_llm_ws'
/content/aps26_llm_ws
remote: Enumerating objects: 5, done.
remote: Counting objects: 100% (5/5), done.
remote: Compressing objects: 100% (3/3), done.
remote: Total 3 (delta 2), reused 0 (delta 0), pack-reused 0 (from 0)
Unpacking objects: 100% (3/3), 1.02 KiB | 1.02 MiB/s, done.
From https://github.com/Kevinoost1/aps26_llm_ws
   929e716..258429a  main       -> origin/main
Updating 929e716..258429a
Fast-forward
 prompt_full.json | 7 ++++++-
 1 file changed, 6 insertions(+), 1 deletion(-)


# Load in the validation DataFrame

The validation dataframe contains only a hand full of labeled examples. For an actual validation set it is recommended to use a lot more

The code just extracts the manual labels and exports the actual transcripts to a .txt, which is the format we are going to use for choicellm

In [25]:
df: pd.DataFrame = pd.read_csv('validation.csv')
y_true = df['manual_alienation']

with open('transcripts.txt', 'w', encoding='utf-8') as f:
    for t in zip(df['translation']):
        t_clean = str(t).replace('\n', ' ').replace('\r', ' ')
        f.write(t_clean + '\n')


# The Current Prompt

Running the code below, will print the starting prompt that we are going to try to improve today.

Note that this prompt has been divided into different parts for convenience:

1. The system prompt tells the model what 'role' it should take. A part of the default system prompt in the ChatGPT models is for example: 'you are a helpful assistant'. This system prompt is supplied during the entire classification process.

2. We have defined two categories for this workshop: 'Alienation' (i.e. us vs. them) rhetoric and 'Other', which is the rest category and will contain any transcripts that do not contain Alienation rhetoric.

3. Later on in this workshop you will encounter a third part of the prompt, namely examples. This turns the classification task from 'zero-shot' classification into 'few-shot' classification. This could massively improve performance if you supply the model with quality examples

4. The user prompt is made up of the categories, examples, and a new transcript for everything it needs to classify.

In [61]:
with open('prompt_full.json') as f:
  prompt = json.load(f)

sys_prompt = prompt['system_prompt']
cat_alienation = prompt['categories']['Alienation']
cat_other = prompt['categories']['Other']

print(f"""
    System Prompt: {sys_prompt}\n
    Definition of Alienation: {cat_alienation}\n
    Definition of Other: {cat_other}\n
    """
)


    System Prompt: U# Main topic

You are a professional text coder of TikTok video transcripts about news, politics & societal issues. Your task is to determine whether the given transcript contains group signaling rehtoric, better known as Alienation, or whether it contains some other type of language.
follow the category definitions below strictly.

{categories}

    Definition of Alienation: Alienates or implies that another individual or group doesn't belong or fit in, uses us vs. them framing

    Definition of Other: The transcript does not contain rhetoric to do with Alienation

    


# Supervised Classification

Now, the actual classification step can begin. We are using a very small toy model for this workshop that does not perform very well out-of the box.

This is because for larger models you either need some very powerful computing resources (GPUs with enough VRAM) or you need to use an API, such as the one by OPENAI. If you are able to run a powerful model locally, just replace the value for the --model argument with the model link on HuggingFace or with a local folder. If you would like to use an API, you can similarly add the --model argument to the model you would like to use (e.g. 'gpt-4o') and add the second argument --openai

For demonstration purposes, we will continue to use the small model. The validation process stays the same

In [36]:
!choicellm transcripts.txt --chat --model 'unsloth/Qwen2.5-7B-Instruct-bnb-4bit' --prompt 'prompt_full.json' > results.csv

args.seed=42023
NumExpr defaulting to 2 threads.
HTTP Request: HEAD https://huggingface.co/unsloth/Qwen2.5-7B-Instruct-bnb-4bit/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/unsloth/Qwen2.5-7B-Instruct-bnb-4bit/bdd404162d94997f390efbfa660eb3f21cbbc81d/config.json "HTTP/1.1 200 OK"
HTTP Request: GET https://huggingface.co/api/resolve-cache/models/unsloth/Qwen2.5-7B-Instruct-bnb-4bit/bdd404162d94997f390efbfa660eb3f21cbbc81d/config.json "HTTP/1.1 200 OK"
config.json: 1.34kB [00:00, 4.34MB/s]
HTTP Request: HEAD https://huggingface.co/unsloth/Qwen2.5-7B-Instruct-bnb-4bit/resolve/main/tokenizer_config.json "HTTP/1.1 307 Temporary Redirect"
HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/unsloth/Qwen2.5-7B-Instruct-bnb-4bit/bdd404162d94997f390efbfa660eb3f21cbbc81d/tokenizer_config.json "HTTP/1.1 200 OK"
HTTP Request: GET https://huggingface.co/api/resolve-cache/models/unsloth/Qwen2.5-7B-Instruct

# Results

With our results retrieved, we can print and inspect if there is anything odd. Likely you will get almost exclusively 'Alienation' predictions due to the model not being very good.

In [37]:
results: pd.DataFrame = pd.read_csv('results.csv')
y_pred = results['pred']

print(y_pred)

0     Alienation
1     Alienation
2     Alienation
3     Alienation
4     Alienation
5     Alienation
6          Other
7     Alienation
8     Alienation
9     Alienation
10    Alienation
11    Alienation
12    Alienation
13    Alienation
14    Alienation
15    Alienation
16    Alienation
17    Alienation
18    Alienation
19    Alienation
20    Alienation
21    Alienation
22    Alienation
23    Alienation
24    Alienation
25    Alienation
26    Alienation
27    Alienation
28    Alienation
29         Other
30    Alienation
31    Alienation
32    Alienation
33    Alienation
34    Alienation
Name: pred, dtype: object


# How does the prompt perform?

We use two main ways of assesing model performance:
- F1 score -> This metric describes a balance between precision and recall

A good F1 score depends on the task, but is generally between 0.7 and 0.9.


- Krippendorff's alpha -> Generally more conservative than F1 score. For validation, try to aim for at least 0.8 at a minimum. However, if you can get it above 0.9 that is even better. For the test set you can just report the metric that comes out. If both your validation set and test set are equally representative of your total data there's a good chance that optimization on the validation set to 0.8 or 0.9 will yield good results on the test set.

Both performance metrics do not care about class imbalance (unlike accuracy). Do keep in mind that with a larger class imbalance, you need a larger test set (it is also good to have a larger validation set in that case)


--------------------------------------------------------------------------------

There is one more metric that I think is very useful, namely the receiver operator characteristic curve (ROC) and its area under the curve (AUC). I highly recommend using this metric since it bases model performance not on the majority label as predicted by the model, but rather on the uncertainty from the model (class probabilities).

However, be careful with using this metric whenever you run a model through an API. It is very likely that the probabilities you get back from the API are not the actual logits from the model, which makes this metric useless.

In [63]:
def get_performance(y_true, y_pred):
  report = classification_report(y_true, y_pred)

  le = LabelEncoder()
  le.fit(pd.concat([y_true, y_pred]))

  kripp_data = np.array([le.transform(y_true), le.transform(y_pred)])
  alpha = krippendorff.alpha(reliability_data=kripp_data, level_of_measurement='nominal')

  print(f'''
  Performance Report:\n\n
  Krippendorff Alpha: {alpha}\n
  Misc. metrics (incl. F1): {report}
  ''')

  return report, alpha

In [64]:
class_report, alpha  = get_performance(y_true, y_pred)


  Performance Report:


  Krippendorff Alpha: -0.3529411764705883

  Misc. metrics (incl. F1):               precision    recall  f1-score   support

  Alienation       0.46      1.00      0.63        16
       Other       0.00      0.00      0.00        19

    accuracy                           0.46        35
   macro avg       0.23      0.50      0.31        35
weighted avg       0.21      0.46      0.29        35

  


/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


# Design your own


Now we can practice a bit with designing our own prompt. You can see the different parts of the prompt below and are allowed to edit them in whichever way you please.

A good way to do this is to edit small parts, then run the classification below, inspect the test scores, and repeat the process until you are satisfied.

Let's see how high you can get the test scores, even if the model is currently not very good.

In [65]:
sys_prompt = f"""
# Main topic\n\n

You are a professional text coder of TikTok video transcripts about news, politics & societal issues. Your task is to determine whether the given transcript contains group signaling rehtoric, better known as Alienation, or whether it contains some other type of language.\n
follow the category definitions below strictly.
"""

In [66]:
cat_alienation = f"""
Alienates or implies that another individual or group doesn't belong or fit in, uses us vs. them framing
"""

In [67]:
cat_other = f"""
The transcript does not contain rhetoric to do with Alienation
"""

In [72]:
def create_new_prompt(sys_prompt, cat_alienation, cat_other, examples=prompt['examples']) -> None:
  new_prompt = prompt
  new_prompt['system_prompt'] = sys_prompt + '\n\n{categories}'
  new_prompt['categories']['Alienation'] = cat_alienation
  new_prompt['categories']['Other'] = cat_other
  new_prompt['examples'] = examples

  with open('new_prompt.json', 'w') as f:
    json.dump(new_prompt, f)

create_new_prompt(sys_prompt, cat_alienation, cat_other)

In [69]:
!choicellm transcripts.txt --model 'unsloth/Qwen2.5-7B-Instruct-bnb-4bit' --prompt 'new_prompt.json' > results.csv

results: pd.DataFrame = pd.read_csv('results.csv')
y_pred = results['pred']

class_report, alpha  = get_performance(y_true, y_pred)

args.seed=76686
NumExpr defaulting to 2 threads.
HTTP Request: HEAD https://huggingface.co/unsloth/Qwen2.5-7B-Instruct-bnb-4bit/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/unsloth/Qwen2.5-7B-Instruct-bnb-4bit/bdd404162d94997f390efbfa660eb3f21cbbc81d/config.json "HTTP/1.1 200 OK"
HTTP Request: HEAD https://huggingface.co/unsloth/Qwen2.5-7B-Instruct-bnb-4bit/resolve/main/tokenizer_config.json "HTTP/1.1 307 Temporary Redirect"
HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/unsloth/Qwen2.5-7B-Instruct-bnb-4bit/bdd404162d94997f390efbfa660eb3f21cbbc81d/tokenizer_config.json "HTTP/1.1 200 OK"
HTTP Request: GET https://huggingface.co/api/models/unsloth/Qwen2.5-7B-Instruct-bnb-4bit/tree/main/additional_chat_templates?recursive=false&expand=false "HTTP/1.1 404 Not Found"
HTTP Request: GET https://huggingface.co/api/models/unsloth/Qwen2.5-7B-Instruct-bnb-4bit/tree/main?recursive=true&expand=false

# Examples

Now, one way you can improve the performance of your prompt drastically is by adding examples to the prompt. Typically you want to inspect your validation set with the predicted labels and see where the model went wrong. The best examples to add are oftentimes transcripts right on the border of two classes (grey area) or things that the model continiously strugles with.

Some potential transcripts you could use as examples can be found in the file 'potential_examples.csv'. This can be downloaded from the GitHub:
https://github.com/Kevinoost1/aps26_llm_ws



In case you would like to add examples, add them in the list, dictionary format below. the "item" is the transcript and the "target_index" is your label. This is zero-based, meaning target_index: 0 = 'Alienation', and target_index: 1 = 'Other'

Just make sure that your examples do NOT appear in the validation set, and especially NOT the test set either.

In [73]:

examples = [
    {
      "item": "what on earth is going on in the house of Commons",
      "target_index": 1
    }
  ]

In [74]:
create_new_prompt(sys_prompt, cat_alienation, cat_other, examples)

!choicellm transcripts.txt --model 'unsloth/Qwen2.5-7B-Instruct-bnb-4bit' --prompt 'new_prompt.json' > results.csv

results: pd.DataFrame = pd.read_csv('results.csv')
y_pred = results['pred']

class_report, alpha  = get_performance(y_true, y_pred)

args.seed=3092
NumExpr defaulting to 2 threads.
HTTP Request: HEAD https://huggingface.co/unsloth/Qwen2.5-7B-Instruct-bnb-4bit/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/unsloth/Qwen2.5-7B-Instruct-bnb-4bit/bdd404162d94997f390efbfa660eb3f21cbbc81d/config.json "HTTP/1.1 200 OK"
HTTP Request: HEAD https://huggingface.co/unsloth/Qwen2.5-7B-Instruct-bnb-4bit/resolve/main/tokenizer_config.json "HTTP/1.1 307 Temporary Redirect"
HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/unsloth/Qwen2.5-7B-Instruct-bnb-4bit/bdd404162d94997f390efbfa660eb3f21cbbc81d/tokenizer_config.json "HTTP/1.1 200 OK"
HTTP Request: GET https://huggingface.co/api/models/unsloth/Qwen2.5-7B-Instruct-bnb-4bit/tree/main/additional_chat_templates?recursive=false&expand=false "HTTP/1.1 404 Not Found"
HTTP Request: GET https://huggingface.co/api/models/unsloth/Qwen2.5-7B-Instruct-bnb-4bit/tree/main?recursive=true&expand=false 

You can from here on out continue to improve all parts of the prompt and add examples as you please.



Remember that this workshop mainly covered the validation part of the process. For a fair assement of your classification performance you need to label a seperate randomly sampled test set and extract performance metrics once with your 'perfect' prompt. This is what you would eventually report.

If you are using an API it might be advisable to run the test classification multiple times and take the average. This is because API-called models are usually the least deterministic in their output.

A further note is that it is possible to correct for your test results in your downstream analyses. If you, for example, wish to run a t-test or regression with your predicted labels, you can look into bootstrapping methods that correct for errors in the model's prediction.